# Analisis Faktor Penyebab Review Buruk pada E-Commerce Olist

**Tujuan:** memahami apa yang membuat pelanggan Olist memberi review buruk (1–2 bintang) dan faktor apa yang paling berpengaruh.

**Data:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), 99.441 pesanan (September 2016 – Oktober 2018). Data digabung terlebih dahulu di BigQuery menjadi tabel `orders_analysis` (satu baris per pesanan), lihat folder `sql/`.

**Rumusan masalah:**
1. Seberapa besar dan bagaimana tren review buruk dari waktu ke waktu?
2. Seberapa besar pengaruh keterlambatan pengiriman terhadap review buruk?
3. Kategori produk, region, dan seller mana yang paling sering mendapat review buruk?
4. Faktor apa lagi yang berpengaruh?

## 1. Persiapan: menarik data dari BigQuery

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!pip install bigquery-magics -q
%load_ext bigquery_magics

In [ ]:
%%bigquery df --project olist-review-analysis-510505
SELECT *
FROM `olist-review-analysis-510505.olist_ecommerce.orders_analysis`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(df.shape)   # harus (99441, 32)
df.info()

Data kosong yang terlihat di `df.info()` masuk akal: 775 pesanan tanpa barang (umumnya dibatalkan), 2.965 pesanan tanpa tanggal sampai, 768 pesanan tanpa review, dan hanya 40.748 review yang berisi komentar.

## 2. Rumusan Masalah 1 — Seberapa besar masalahnya?

In [ ]:
df_rev = df[df['review_score'].notna()].copy()

print("Jumlah pesanan dengan review:", len(df_rev))
print("Persentase review buruk:", round(df_rev['review_buruk'].mean() * 100, 2), "%")
print()
print(df_rev['review_score'].value_counts().sort_index())

In [ ]:
status = df_rev.groupby('order_status')['review_buruk'].agg(['size', 'mean'])
status.columns = ['jumlah_pesanan', 'pct_review_buruk']
status['pct_review_buruk'] = (status['pct_review_buruk'] * 100).round(1)
status.sort_values('jumlah_pesanan', ascending=False)

Pesanan yang tidak sampai ke pelanggan (shipped, canceled, unavailable, invoiced, dll.) hanya sekitar 3% dari pesanan, tetapi review buruknya 70–93% dan menyumbang sekitar 15% dari seluruh review buruk. Status seperti *shipped* dan *invoiced* ternyata pesanan yang macet berbulan-bulan, bukan yang masih di jalan.

**Keputusan:** analisis berikutnya fokus pada pesanan *delivered*.

In [ ]:
df_rev['bulan'] = df_rev['order_purchase_timestamp'].dt.tz_localize(None).dt.to_period('M')

tren = df_rev.groupby('bulan').agg(
    jumlah_pesanan=('review_buruk', 'size'),
    pct_review_buruk=('review_buruk', 'mean')
)
tren['pct_review_buruk'] = tren['pct_review_buruk'] * 100
tren = tren[tren['jumlah_pesanan'] >= 100]   # buang bulan dengan pesanan terlalu sedikit

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(tren.index.astype(str), tren['jumlah_pesanan'], color='lightgray')
ax1.set_ylabel('Jumlah pesanan')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(tren.index.astype(str), tren['pct_review_buruk'], color='crimson', marker='o')
ax2.set_ylabel('% review buruk')

plt.title('Jumlah Pesanan dan Persentase Review Buruk per Bulan')
fig.tight_layout()
plt.show()

**Temuan:** 14,7% pesanan mendapat review buruk. Lonjakan terjadi pada November 2017 dan Februari–Maret 2018. Ramainya pesanan bukan penyebabnya, karena April–Agustus 2018 tetap ramai tetapi review buruknya rendah.

## 3. Rumusan Masalah 2 — Pengaruh keterlambatan

In [ ]:
df_del = df_rev[
    (df_rev['order_status'] == 'delivered') &
    (df_rev['terlambat'].notna())
].copy()

print("Jumlah pesanan delivered:", len(df_del))
print("Persentase terlambat:", round(df_del['terlambat'].mean() * 100, 2), "%")

ringkas = df_del.groupby('terlambat').agg(
    jumlah_pesanan=('review_buruk', 'size'),
    pct_review_buruk=('review_buruk', 'mean'),
    rata_rata_bintang=('review_score', 'mean')
)
ringkas['pct_review_buruk'] = (ringkas['pct_review_buruk'] * 100).round(1)
ringkas['rata_rata_bintang'] = ringkas['rata_rata_bintang'].round(2)
ringkas

In [ ]:
bins = [-1000, -15, -8, -1, 0, 3, 7, 14, 1000]
labels = ['>14 hari\nlebih cepat', '8-14 hari\nlebih cepat', '1-7 hari\nlebih cepat',
          'Tepat di hari\nestimasi', 'Telat\n1-3 hari', 'Telat\n4-7 hari',
          'Telat\n8-14 hari', 'Telat\n>14 hari']

df_del['kelompok_selisih'] = pd.cut(
    df_del['selisih_dari_estimasi_hari'].astype(float),
    bins=bins, labels=labels
)

per_kelompok = df_del.groupby('kelompok_selisih', observed=True)['review_buruk'].agg(['size', 'mean'])
per_kelompok.columns = ['jumlah_pesanan', 'pct_review_buruk']
per_kelompok['pct_review_buruk'] = (per_kelompok['pct_review_buruk'] * 100).round(1)
print(per_kelompok)

warna = ['seagreen'] * 3 + ['gray'] + ['crimson'] * 4
plt.figure(figsize=(12, 5))
plt.bar(per_kelompok.index.astype(str), per_kelompok['pct_review_buruk'].astype(float), color=warna)
plt.axhline(df_del['review_buruk'].mean() * 100, color='black', linestyle='--',
            label='Rata-rata semua pesanan delivered')
plt.ylabel('% review buruk')
plt.title('Persentase Review Buruk Berdasarkan Selisih dari Tanggal Estimasi')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_del['bulan'] = df_del['order_purchase_timestamp'].dt.tz_localize(None).dt.to_period('M')

tren_kirim = df_del.groupby('bulan').agg(
    jumlah_pesanan=('review_buruk', 'size'),
    pct_terlambat=('terlambat', 'mean'),
    pct_review_buruk=('review_buruk', 'mean')
)
tren_kirim = tren_kirim[tren_kirim['jumlah_pesanan'] >= 100]
tren_kirim[['pct_terlambat', 'pct_review_buruk']] = tren_kirim[['pct_terlambat', 'pct_review_buruk']].astype(float) * 100

plt.figure(figsize=(12, 5))
plt.plot(tren_kirim.index.astype(str), tren_kirim['pct_terlambat'], marker='o', color='darkorange', label='% pesanan terlambat')
plt.plot(tren_kirim.index.astype(str), tren_kirim['pct_review_buruk'], marker='o', color='crimson', label='% review buruk')
plt.xticks(rotation=45)
plt.ylabel('Persen')
plt.title('Persentase Keterlambatan dan Review Buruk per Bulan (Pesanan Delivered)')
plt.legend()
plt.tight_layout()
plt.show()

print("Korelasi keterlambatan vs review buruk per bulan:",
      round(tren_kirim['pct_terlambat'].corr(tren_kirim['pct_review_buruk']), 3))

In [ ]:
buruk = df_del[df_del['review_buruk'] == True]
print("Jumlah review buruk (delivered):", len(buruk))
print("Dari pesanan terlambat:", round(buruk['terlambat'].mean() * 100, 1), "%")
print("Dari pesanan tepat waktu:", round((1 - buruk['terlambat'].mean()) * 100, 1), "%")

**Temuan:**
- Pesanan terlambat punya peluang review buruk hampir 7 kali lipat (62,4% vs 9,3%).
- Pelanggan menilai apakah janji tanggal ditepati, bukan seberapa cepat barang datang. Telat 1–3 hari saja sudah 32%, dan mentok di sekitar 80% setelah seminggu.
- Lonjakan review buruk mengikuti lonjakan keterlambatan (korelasi bulanan 0,81 dari 21 bulan).
- Namun keterlambatan hanya menjelaskan sekitar sepertiga review buruk. Dua pertiga sisanya berasal dari pesanan yang tepat waktu.

## 4. Rumusan Masalah 3 — Region, kategori, dan seller

In [ ]:
df_ontime = df_del[df_del['terlambat'] == False].copy()
print("Pesanan tepat waktu:", len(df_ontime))
print("Review buruk pada pesanan tepat waktu:", round(df_ontime['review_buruk'].mean() * 100, 2), "%")

def ringkas_per(data, kolom, min_pesanan=300):
    hasil = data.groupby(kolom).agg(
        jumlah_pesanan=('review_buruk', 'size'),
        jumlah_review_buruk=('review_buruk', 'sum'),
        pct_review_buruk=('review_buruk', 'mean')
    )
    hasil = hasil[hasil['jumlah_pesanan'] >= min_pesanan].copy()
    hasil['pct_review_buruk'] = (hasil['pct_review_buruk'] * 100).round(1)
    return hasil.sort_values('pct_review_buruk', ascending=False)

In [ ]:
region = df_del.groupby('customer_state').agg(
    jumlah_pesanan=('review_buruk', 'size'),
    pct_review_buruk=('review_buruk', 'mean'),
    pct_terlambat=('terlambat', 'mean')
)
region = region[region['jumlah_pesanan'] >= 300].copy()
region[['pct_review_buruk', 'pct_terlambat']] = (region[['pct_review_buruk', 'pct_terlambat']] * 100).round(1)

region_ontime = ringkas_per(df_ontime, 'customer_state', min_pesanan=0)
gabung = region.join(region_ontime['pct_review_buruk'].rename('pct_buruk_tepat_waktu'))
gabung = gabung.sort_values('pct_review_buruk', ascending=False)
gabung = gabung.astype(float)
print(gabung)

y = np.arange(len(gabung))
plt.figure(figsize=(10, 8))
plt.barh(y - 0.2, gabung['pct_review_buruk'], height=0.4, color='crimson', label='Semua pesanan delivered')
plt.barh(y + 0.2, gabung['pct_buruk_tepat_waktu'], height=0.4, color='steelblue', label='Hanya pesanan tepat waktu')
plt.yticks(y, gabung.index)
plt.gca().invert_yaxis()
plt.xlabel('% review buruk')
plt.title('Review Buruk per State Pelanggan')
plt.legend()
plt.tight_layout()
plt.show()

print("Korelasi keterlambatan vs review buruk antar state:",
      round(gabung['pct_terlambat'].corr(gabung['pct_review_buruk']), 3))

In [ ]:
kategori = ringkas_per(df_ontime, 'kategori_utama').astype(float)
print("Jumlah kategori dengan minimal 300 pesanan:", len(kategori))
print("\n10 kategori dengan persentase review buruk tertinggi:")
print(kategori.head(10))
print("\n5 kategori dengan persentase review buruk terendah:")
print(kategori.tail(5))

rata2 = df_ontime['review_buruk'].mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_pct = kategori.head(10)
axes[0].barh(top_pct.index, top_pct['pct_review_buruk'], color='crimson')
axes[0].axvline(rata2, color='black', linestyle='--', label='Rata-rata')
axes[0].invert_yaxis()
axes[0].set_title('Persentase Review Buruk Tertinggi')
axes[0].set_xlabel('% review buruk')
axes[0].legend()

top_jumlah = kategori.sort_values('jumlah_review_buruk', ascending=False).head(10)
axes[1].barh(top_jumlah.index, top_jumlah['jumlah_review_buruk'], color='darkorange')
axes[1].invert_yaxis()
axes[1].set_title('Jumlah Review Buruk Terbanyak')
axes[1].set_xlabel('Jumlah review buruk')

plt.suptitle('Kategori Produk pada Pesanan Tepat Waktu')
plt.tight_layout()
plt.show()

In [ ]:
seller = ringkas_per(df_ontime, 'seller_utama', min_pesanan=50).astype(float)
print("Seller dengan minimal 50 pesanan tepat waktu:", len(seller))
print(seller['pct_review_buruk'].describe().round(1))

plt.figure(figsize=(10, 5))
plt.hist(seller['pct_review_buruk'], bins=30, color='steelblue', edgecolor='white')
plt.axvline(20, color='crimson', linestyle='--', label='Batas seller bermasalah (20%)')
plt.xlabel('% review buruk per seller')
plt.ylabel('Jumlah seller')
plt.title('Sebaran Persentase Review Buruk per Seller')
plt.legend()
plt.tight_layout()
plt.show()

bermasalah = seller[seller['pct_review_buruk'] >= 20]
total_buruk = df_ontime['review_buruk'].sum()
print("Jumlah seller bermasalah (>= 20% review buruk):", len(bermasalah))
print("Review buruk dari seller bermasalah:", int(bermasalah['jumlah_review_buruk'].sum()))
print("Porsi dari total review buruk tepat waktu:",
      round(bermasalah['jumlah_review_buruk'].sum() / total_buruk * 100, 1), "%")

**Temuan:**
- **Region:** perbedaan antar state hampir seluruhnya disebabkan keterlambatan. Saat tepat waktu, semua state berada di kisaran 7–11%. Wilayah timur laut (AL, MA, SE, CE) dan RJ paling terdampak.
- **Kategori:** furnitur dan produk rumah tangga paling berisiko, terutama office_furniture (18,5%). bed_bath_table, computers_accessories, dan furniture_decor jadi prioritas karena persentase dan volumenya sama-sama tinggi.
- **Seller:** masalahnya sistemik. 19 seller bermasalah hanya menyumbang 8,2% review buruk pada pesanan tepat waktu.

## 5. Rumusan Masalah 4 — Faktor lainnya

In [ ]:
faktor = {
    'jumlah_barang': ([0, 1, 2, 3, 100], ['1', '2', '3', '4+'], 'Jumlah barang per pesanan'),
    'jumlah_seller': ([0, 1, 2, 100], ['1', '2', '3+'], 'Jumlah seller per pesanan'),
    'berat_barang_utama_g': ([-1, 500, 2000, 5000, 15000, 1e9], ['<0.5kg', '0.5-2kg', '2-5kg', '5-15kg', '>15kg'], 'Berat barang utama'),
    'total_harga': ([0, 50, 100, 200, 500, 1e9], ['<50', '50-100', '100-200', '200-500', '>500'], 'Total harga (R$)'),
    'rasio_ongkir': ([-0.01, 0.1, 0.2, 0.4, 0.7, 1e9], ['<10%', '10-20%', '20-40%', '40-70%', '>70%'], 'Rasio ongkir terhadap harga'),
    'jumlah_cicilan': ([-1, 1, 3, 6, 10, 100], ['1x', '2-3x', '4-6x', '7-10x', '>10x'], 'Jumlah cicilan'),
}

rata2 = df_ontime['review_buruk'].mean() * 100
# sharey=True: semua grafik memakai skala yang sama agar kekuatan faktor bisa dibandingkan dengan jujur
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True)
axes = axes.flatten()

for ax, (kolom, (bins, labels, judul)) in zip(axes, faktor.items()):
    kelompok = pd.cut(df_ontime[kolom].astype(float), bins=bins, labels=labels)
    hasil = df_ontime.groupby(kelompok, observed=True)['review_buruk'].agg(['size', 'mean'])
    hasil.columns = ['jumlah_pesanan', 'pct_review_buruk']
    hasil = hasil[hasil['jumlah_pesanan'] >= 50].astype(float)
    hasil['pct_review_buruk'] = (hasil['pct_review_buruk'] * 100).round(1)

    print(f"\n== {judul}")
    print(hasil)

    ax.bar(hasil.index.astype(str), hasil['pct_review_buruk'], color='steelblue')
    ax.axhline(rata2, color='crimson', linestyle='--')
    ax.set_title(judul)
    ax.set_ylabel('% review buruk')

plt.suptitle('Persentase Review Buruk per Faktor (Pesanan Tepat Waktu)\nGaris merah = rata-rata', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print("== Metode bayar utama")
print(ringkas_per(df_ontime, 'metode_bayar_utama', min_pesanan=50))

print("\n== Seller dan pelanggan di state yang sama?")
print(ringkas_per(df_ontime, 'satu_state', min_pesanan=50))

In [ ]:
jb = df_ontime['jumlah_barang'].fillna(0).astype(int)
js = df_ontime['jumlah_seller'].fillna(0).astype(int)

df_ontime['tipe_pesanan'] = np.select(
    [jb == 1, (jb > 1) & (js == 1), js > 1],
    ['1 barang', '>1 barang, 1 seller', '>1 barang, >1 seller'],
    default='lainnya'
)

tipe = df_ontime.groupby('tipe_pesanan')['review_buruk'].agg(['size', 'mean', 'sum'])
tipe.columns = ['jumlah_pesanan', 'pct_review_buruk', 'jumlah_review_buruk']
tipe['pct_dari_semua_pesanan'] = (tipe['jumlah_pesanan'] / tipe['jumlah_pesanan'].sum() * 100).round(1)
tipe['pct_dari_semua_review_buruk'] = (tipe['jumlah_review_buruk'] / tipe['jumlah_review_buruk'].sum() * 100).round(1)
tipe['pct_review_buruk'] = (tipe['pct_review_buruk'] * 100).round(1)
tipe

In [ ]:
# Kata kunci bahasa Portugis untuk keluhan barang tidak lengkap:
# "não recebi" (tidak menerima), "recebi apenas/só" (hanya menerima),
# "falta/faltando" (kurang), "incompleto", "um dos produtos" (salah satu produk)
pola = r'(?i)(?:n[aã]o receb|recebi (?:apenas|s[oó]|somente)|falt|um dos produtos|uma das|apenas um|s[oó] um|incomplet|veio s[oó])'

buruk_komentar = df_ontime[
    (df_ontime['review_buruk'] == True) &
    (df_ontime['komentar_review'].notna())
].copy()
buruk_komentar['keluhan_tidak_lengkap'] = buruk_komentar['komentar_review'].str.contains(pola, regex=True)

hasil = buruk_komentar.groupby('tipe_pesanan')['keluhan_tidak_lengkap'].agg(['size', 'mean'])
hasil.columns = ['review_buruk_berkomentar', 'pct_keluhan_tidak_lengkap']
hasil['pct_keluhan_tidak_lengkap'] = (hasil['pct_keluhan_tidak_lengkap'] * 100).round(1)
print(hasil)

print("\nContoh komentar dari pesanan '>1 barang, >1 seller':")
contoh = buruk_komentar[
    (buruk_komentar['tipe_pesanan'] == '>1 barang, >1 seller') &
    (buruk_komentar['keluhan_tidak_lengkap'])
]['komentar_review'].head(5)
for k in contoh:
    print("-", k)

**Temuan:**
- Faktor terkuat setelah keterlambatan adalah **kelengkapan pesanan**. Pesanan berisi lebih dari satu barang hanya 10% dari pesanan tepat waktu, tetapi menyumbang 27% review buruk. Pesanan dari beberapa seller punya peluang review buruk 46,9%.
- Keluhan "barang tidak lengkap" muncul di 13% review buruk pesanan 1 barang, tetapi 40–56% pada pesanan multi-barang. Dugaan kuat: pesanan tercatat sampai saat paket pertama tiba, padahal barang lain belum diterima. Ini dugaan berdasarkan komentar, karena data tidak mencatat tanggal sampai per paket.
- Berat, harga, dan cicilan berpengaruh kecil. Ongkir, metode bayar, dan lokasi seller hampir tidak berpengaruh.

## 6. Kesimpulan

Review buruk Olist berasal dari tiga sumber utama:

1. **Pesanan tidak sampai**: sekitar 3% pesanan, tetapi 15% review buruk.
2. **Pesanan terlambat**: 27% review buruk, dengan lonjakan tajam begitu melewati tanggal estimasi.
3. **Pesanan multi-barang yang tidak lengkap**: 15% review buruk, walaupun tercatat tepat waktu.

Kategori produk dan seller bermasalah hanya menambah sedikit penjelasan. Sekitar 42% review buruk berasal dari pesanan 1 barang yang tepat waktu, dan penyebabnya (misalnya kualitas produk) tidak tercatat di data.

**Rekomendasi:**
- Peringatan otomatis untuk pesanan yang melewati estimasi tanpa status sampai.
- Estimasi tanggal yang realistis per rute, dimulai dari wilayah timur laut dan RJ.
- Status "sampai" hanya setelah semua paket diterima, dan informasi pengiriman terpisah sejak awal.
- Standar kemasan dan deskripsi produk untuk kategori furnitur.
- Evaluasi 19 seller dengan review buruk minimal 20%.